<a href="https://colab.research.google.com/github/heyitsarun/Aero-Gen-Synthetic-Anomaly-Generation-in-Aviation-Communications/blob/main/01_data_prep_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/aerogen_project'
CACHE_DIR = f'{PROJECT_ROOT}/cache'
DATA_DIR = f'{PROJECT_ROOT}/data'

for d in [PROJECT_ROOT, CACHE_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Project folders ready at:', PROJECT_ROOT)

Mounted at /content/drive
Project folders ready at: /content/drive/MyDrive/aerogen_project


In [2]:
!pip install -q datasets pandas

README.md:   0%|          | 0.00/6.03k [00:00<?, ?B/s]

data/train-00000-of-00004-c1d7fb31dcbf64(…):   0%|          | 0.00/488M [00:00<?, ?B/s]

data/train-00001-of-00004-f165730df6bf72(…):   0%|          | 0.00/468M [00:00<?, ?B/s]

data/train-00002-of-00004-67e682f17e32b7(…):   0%|          | 0.00/495M [00:00<?, ?B/s]

data/train-00003-of-00004-b0b05d4b243c95(…):   0%|          | 0.00/474M [00:00<?, ?B/s]

data/test-00000-of-00001-01544bdf54b4ccf(…):   0%|          | 0.00/470M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7638 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1901 [00:00<?, ? examples/s]

Cached to Drive: /content/drive/MyDrive/aerogen_project/cache/atcosim_raw.csv
(9539, 6)


,id,text,segment_start_time,segment_end_time,duration,split
0,atcosim_sm1_01_001_000000_000329,psa eight one zero turn right to trasadingen,0.0,3.30,3.30,train
1,atcosim_sm1_01_002_000000_000466,lufthansa five three one eight contact zurich ...,0.0,4.66,4.66,train
2,atcosim_sm1_01_003_000000_000422,psa eight one zero contact zurich one three th...,0.0,4.23,4.23,train
3,atcosim_sm1_01_004_000000_000264,sabena four eight one rhein identified,0.0,2.64,2.64,train
4,atcosim_sm1_01_005_000000_000391,transwede one zero one rhein identified set co...,0.0,3.91,3.91,train


In [5]:
atcosim_df['text'] = atcosim_df['text'].astype(str).str.strip().str.lower()
atcosim_df['n_tokens'] = atcosim_df['text'].str.split().apply(len)

print('Token length distribution:')
print(atcosim_df['n_tokens'].describe())

MIN_TOKENS = 4
clean_df = atcosim_df[atcosim_df['n_tokens'] >= MIN_TOKENS].reset_index(drop=True)
print(f'Kept {len(clean_df)} / {len(atcosim_df)} rows after dropping segments under {MIN_TOKENS} tokens')

Token length distribution:
count    9539.000000
mean       11.257994
std         4.152246
min         1.000000
25%         9.000000
50%        11.000000
75%        13.000000
max        71.000000
Name: n_tokens, dtype: float64
Kept 9204 / 9539 rows after dropping segments under 4 tokens


In [10]:
NUMBER_WORDS = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9'
}

PHONETIC_ALPHABET = {
    'alpha','bravo','charlie','delta','echo','foxtrot','golf','hotel','india',
    'juliett','juliet','kilo','lima','mike','november','oscar','papa','quebec',
    'romeo','sierra','tango','uniform','victor','whiskey','xray','yankee','zulu'
}

FILLER_WORDS = {'is', 'are', 'you', 'good', 'morning', 'afternoon', 'evening'}

NON_INSTRUCTION_STARTERS = {
    'negative', 'affirm', 'affirmative', 'roger', 'what', 'how', 'say',
    'wilco', 'copy', 'standby', 'unable', 'confirm'
}

CALLSIGN_TOKEN_CAP = 6  # generous upper bound; longer unmatched lines get flagged, not guessed

def split_callsign_instruction_v2(text):
    tokens = text.split()
    if not tokens:
        return '', '', 'empty'

    if tokens[0] in NON_INSTRUCTION_STARTERS:
        return '', text, 'non_instruction'

    callsign_tokens = []
    seen_identifier = False  # True once we've consumed a number-word or phonetic-alphabet token
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        remainder = ' '.join(tokens[i:])
        is_kw = any(remainder.startswith(kw) for kw in INSTRUCTION_KEYWORDS)

        if is_kw and seen_identifier:
            break  # real instruction verb after a genuine callsign — stop here
        if tok in FILLER_WORDS and seen_identifier:
            break  # filler word after a genuine callsign — stop here, don't consume it

        callsign_tokens.append(tok)
        if tok in NUMBER_WORDS or tok in PHONETIC_ALPHABET:
            seen_identifier = True
        i += 1
        if len(callsign_tokens) >= CALLSIGN_TOKEN_CAP:
            break

    callsign = ' '.join(callsign_tokens)
    instruction = ' '.join(tokens[i:])

    if not instruction and not seen_identifier:
        utterance_type = 'ambiguous_no_keyword'  # flag for manual review, don't pretend we parsed it
    elif not instruction:
        utterance_type = 'callsign_only'
    else:
        utterance_type = 'instruction'

    return callsign, instruction, utterance_type

# Run the parser
clean_df[['callsign', 'instruction', 'utterance_type']] = clean_df['text'].apply(
    lambda t: pd.Series(split_callsign_instruction_v2(t))
)

print(clean_df['utterance_type'].value_counts())
display(clean_df[['text', 'callsign', 'instruction', 'utterance_type']].sample(15, random_state=42))

# Flag and exclude rows that hit the token cap (uncertain parse)
clean_df['hit_cap'] = clean_df['callsign'].apply(lambda c: len(c.split()) >= CALLSIGN_TOKEN_CAP)

print(f"\n{clean_df['hit_cap'].sum()} / {len(clean_df)} rows hit the token cap (uncertain parse — exclude from generation)")

# Final usable set for Stage 2: confident instruction/callsign_only parses only
usable_df = clean_df[
    (clean_df['utterance_type'].isin(['instruction', 'callsign_only'])) & (~clean_df['hit_cap'])
].reset_index(drop=True)
print(f"Usable rows for generation: {len(usable_df)} / {len(clean_df)}")

utterance_type
instruction             8650
non_instruction          269
callsign_only            193
ambiguous_no_keyword      92
Name: count, dtype: int64


,text,callsign,instruction,utterance_type
76,psa five two six hotel rhein radar identified,psa five two six hotel rhein,radar identified,instruction
5281,negative cleared level two nine zero,,negative cleared level two nine zero,non_instruction
2087,saudia one zero six contact rhein radar one th...,saudia one zero six,contact rhein radar one three two decimal four,instruction
582,golf bravo victor juliett india contact zurich...,golf bravo victor juliett india,contact zurich one three three decimal four,instruction
93,alitalia four zero one climb flight level two ...,alitalia four zero one,climb flight level two nine zero,instruction
5148,saudia one five five you are identified cleare...,saudia one five five,you are identified cleared st prex dijon fligh...,instruction
3487,alitalia one three five is identified good aft...,alitalia one three five,is identified good afternoon,instruction
7076,japanair four one nine descend to flight level...,japanair four one nine,descend to flight level two five zero to cross...,instruction
7337,speed way three four six nine descend to fligh...,speed way three four six nine,descend to flight level two seven zero,instruction
1022,hapag lloyd six five three rhein,hapag lloyd six five three rhein,,callsign_only



3648 / 9204 rows hit the token cap (uncertain parse — exclude from generation)
Usable rows for generation: 5216 / 9204


In [11]:
cap_hit_sample = clean_df[clean_df['hit_cap']].sample(15, random_state=1)
display(cap_hit_sample[['text', 'callsign', 'instruction']])

# How many cap-hit rows contain zero instruction keywords anywhere in the full text?
no_kw_at_all = clean_df[clean_df['hit_cap']]['text'].apply(
    lambda t: not any(kw in t for kw in INSTRUCTION_KEYWORDS)
)
print(f"{no_kw_at_all.sum()} / {clean_df['hit_cap'].sum()} cap-hit rows contain no instruction keyword anywhere")

,text,callsign,instruction
7722,alitalia six four seven four own navigation tr...,alitalia six four seven four own,navigation trasadingen karlsruhe
8349,reach six zero one five three contact rhein on...,reach six zero one five three,contact rhein one three two decimal four
1025,german air force five eight five non rvsm in r...,german air force five eight five,non rvsm in radar contact
7063,tarom seven five three four zurich radar good ...,tarom seven five three four zurich,radar good morning identified trasadingen zuri...
3571,constellation two two five one guten tag ident...,constellation two two five one guten,tag identified
5496,good morning trans avia three eight one radar ...,good morning trans avia three eight,one radar contact maintain flight level three ...
4812,alitalia three seven nine rate of descent two ...,alitalia three seven nine rate of,descent two thousand feet per minute
4559,portugal four one one three bonjour i ll call ...,portugal four one one three bonjour,i ll call you back
5275,air france three two six one good morning roge...,air france three two six one,good morning roger squawk ident
6263,air france three five six reims one three four...,air france three five six reims,one three four decimal four good bye


1522 / 3648 cap-hit rows contain no instruction keyword anywhere


In [12]:
def split_callsign_instruction_v3(text):
    tokens = text.split()
    if not tokens:
        return '', '', 'empty', False

    if tokens[0] in NON_INSTRUCTION_STARTERS:
        return '', text, 'non_instruction', False

    callsign_tokens = []
    seen_identifier = False
    capped = False  # True only if we stopped because of the cap, not a real boundary
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        remainder = ' '.join(tokens[i:])
        is_kw = any(remainder.startswith(kw) for kw in INSTRUCTION_KEYWORDS)

        if is_kw and seen_identifier:
            break
        if tok in FILLER_WORDS and seen_identifier:
            break

        if len(callsign_tokens) >= CALLSIGN_TOKEN_CAP:
            capped = True
            break  # stop BEFORE appending — this token was never evaluated as a real boundary

        callsign_tokens.append(tok)
        if tok in NUMBER_WORDS or tok in PHONETIC_ALPHABET:
            seen_identifier = True
        i += 1

    callsign = ' '.join(callsign_tokens)
    instruction = ' '.join(tokens[i:])

    if not instruction and not seen_identifier:
        utterance_type = 'ambiguous_no_keyword'
    elif not instruction:
        utterance_type = 'callsign_only'
    else:
        utterance_type = 'instruction'

    return callsign, instruction, utterance_type, capped

clean_df[['callsign', 'instruction', 'utterance_type', 'hit_cap']] = clean_df['text'].apply(
    lambda t: pd.Series(split_callsign_instruction_v3(t))
)

print(clean_df['utterance_type'].value_counts())
print(f"\n{clean_df['hit_cap'].sum()} / {len(clean_df)} rows genuinely hit the cap without finding a real boundary")

usable_df = clean_df[
    (clean_df['utterance_type'].isin(['instruction', 'callsign_only'])) & (~clean_df['hit_cap'])
].reset_index(drop=True)
print(f"Usable rows for generation: {len(usable_df)} / {len(clean_df)}")

utterance_type
instruction             8650
non_instruction          269
callsign_only            193
ambiguous_no_keyword      92
Name: count, dtype: int64

2324 / 9204 rows genuinely hit the cap without finding a real boundary
Usable rows for generation: 6519 / 9204


In [13]:
PARSED_OUTPUT_PATH = f'{DATA_DIR}/atcosim_parsed.csv'
usable_df.to_csv(PARSED_OUTPUT_PATH, index=False)

print('Saved parsed + filtered ATCOSIM to:', PARSED_OUTPUT_PATH)
print(f'{len(usable_df)} usable rows ready for Stage 2')

Saved parsed + filtered ATCOSIM to: /content/drive/MyDrive/aerogen_project/data/atcosim_parsed.csv
6519 usable rows ready for Stage 2


In [14]:
NUMBER_WORDS = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9'
}

def extract_number_words(text):
    return [tok for tok in text.split() if tok in NUMBER_WORDS]

usable_df['number_tokens'] = usable_df['instruction'].apply(extract_number_words)
usable_df['has_numbers'] = usable_df['number_tokens'].apply(lambda x: len(x) > 0)

print(f"{usable_df['has_numbers'].sum()} / {len(usable_df)} rows contain digit-word sequences (candidates for readback-error injection)")
usable_df[usable_df['has_numbers']][['text', 'number_tokens']].head(10)

4346 / 6519 rows contain digit-word sequences (candidates for readback-error injection)


,text,number_tokens
1,lufthansa five three one eight contact zurich ...,"[one, three, four, six]"
2,psa eight one zero contact zurich one three th...,"[one, three, three, four]"
5,india oscar kilo contact rhein one two seven t...,"[one, two, seven, three, seven]"
8,swiss air nine three five two climb flight lev...,"[three, five, zero]"
9,lufthansa eight two six eight rhein identified...,"[two, seven, zero]"
11,speed bird one five six contact rhein one two ...,"[one, two, seven, three, seven]"
12,foxtrot sierra india rhein identified climb fl...,"[two, six, zero]"
16,sabena four eight one contact rhein one three ...,"[one, three, two, four]"
17,india sierra alfa turn one five degrees to the...,"[one, five]"
18,hapag lloyd six five three rhein identified cl...,"[two, seven, zero]"


In [18]:
ASRS_PATH = f'{DATA_DIR}/asrs_clean_taxonomy.csv'

if not os.path.exists(ASRS_PATH):
    print('asrs_clean_taxonomy.csv not found in Drive — upload it now:')
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    import shutil
    shutil.move(fname, ASRS_PATH)

asrs_df = pd.read_csv(ASRS_PATH)
print(asrs_df['Category'].value_counts())
asrs_df.head()

asrs_clean_taxonomy.csv not found in Drive — upload it now:


Saving asrs_clean_taxonomy.csv to asrs_clean_taxonomy.csv
Category
CC      40
BOTH     3
RB       2
Name: count, dtype: int64


,ACN,Category,Primary Problem,Synopsis
0,835922,BOTH,Human Factors,An air carrier crew reported how a call sign s...
1,842792,BOTH,Human Factors,An IFR single pilot had an altitude deviation ...
2,848854,CC,Procedure,N90 Controller expressed concern regarding som...
3,849298,CC,Human Factors,Air carrier at FL330 with four digit flight nu...
4,851817,CC,Procedure,Air carrier with ZTL described altitude assign...


In [19]:
PARSED_OUTPUT_PATH = f'{DATA_DIR}/atcosim_parsed.csv'
usable_df.to_csv(PARSED_OUTPUT_PATH, index=False)

print('Saved parsed + filtered ATCOSIM to:', PARSED_OUTPUT_PATH)
print(f'{len(usable_df)} usable rows ready for Stage 2')
print('Saved ASRS taxonomy at:', ASRS_PATH)

Saved parsed + filtered ATCOSIM to: /content/drive/MyDrive/aerogen_project/data/atcosim_parsed.csv
6519 usable rows ready for Stage 2
Saved ASRS taxonomy at: /content/drive/MyDrive/aerogen_project/data/asrs_clean_taxonomy.csv


In [22]:
INSTRUCTION_TYPE_MAP = {
    'climb': 'CLIMB', 'descend': 'DESCEND', 'turn': 'TURN',
    'contact': 'CONTACT', 'maintain': 'MAINTAIN', 'cleared': 'CLEARANCE',
    'identified': 'IDENTIFY', 'squawk': 'SQUAWK', 'report': 'REPORT',
    'proceed': 'PROCEED', 'hold': 'HOLD', 'taxi': 'TAXI',
    'reduce': 'SPEED_CHANGE', 'increase': 'SPEED_CHANGE',
    'cross': 'CROSS', 'set course': 'SET_COURSE', 'fly heading': 'TURN',
}

def classify_instruction_type(instruction):
    for kw, label in INSTRUCTION_TYPE_MAP.items():
        if instruction.startswith(kw) or f' {kw} ' in f' {instruction} ':
            return label
    return 'OTHER'

def classify_number_context(instruction, instruction_type):
    """Lightweight rule: value type inferred from the instruction type it follows.
    Not perfect, but consistent and defensible — document as a design choice."""
    if instruction_type in ('CLIMB', 'DESCEND', 'MAINTAIN'):
        return 'altitude'
    if instruction_type == 'CONTACT':
        return 'frequency'
    if instruction_type in ('TURN', 'SET_COURSE'):
        return 'heading'
    if instruction_type == 'SQUAWK':
        return 'squawk_code'
    if 'runway' in instruction:
        return 'runway'
    return 'unclassified'

usable_df['instruction_type'] = usable_df['instruction'].apply(classify_instruction_type)
usable_df['value_type'] = usable_df.apply(
    lambda r: classify_number_context(r['instruction'], r['instruction_type']), axis=1
)

print(usable_df['instruction_type'].value_counts())
print()
print(usable_df['value_type'].value_counts())

instruction_type
CONTACT         2064
CLIMB           1191
IDENTIFY         749
DESCEND          589
TURN             532
OTHER            410
SQUAWK           259
PROCEED          195
MAINTAIN         173
CLEARANCE        125
REPORT           111
SET_COURSE       107
SPEED_CHANGE      11
CROSS              3
Name: count, dtype: int64

value_type
frequency       2064
altitude        1953
unclassified    1604
heading          639
squawk_code      259
Name: count, dtype: int64


In [23]:
def confidence_tier(row):
    if row['utterance_type'] == 'instruction' and not row['hit_cap'] and row['instruction_type'] != 'OTHER':
        return 'high'
    if row['utterance_type'] in ('instruction', 'callsign_only') and not row['hit_cap']:
        return 'medium'
    return 'low'

usable_df['parser_confidence'] = usable_df.apply(confidence_tier, axis=1)
print(usable_df['parser_confidence'].value_counts())

parser_confidence
high      6109
medium     410
Name: count, dtype: int64


In [24]:
FINAL_ENRICHED_PATH = f'{DATA_DIR}/atcosim_enriched.csv'
usable_df.to_csv(FINAL_ENRICHED_PATH, index=False)
print(f'Saved enriched dataset: {len(usable_df)} rows -> {FINAL_ENRICHED_PATH}')

Saved enriched dataset: 6519 rows -> /content/drive/MyDrive/aerogen_project/data/atcosim_enriched.csv


In [25]:
import json

data_dictionary = {
    "dataset": "atcosim_enriched.csv",
    "source": "ATCOSIM corpus (Jzuluaga/atcosim_corpus, HuggingFace)",
    "n_rows": len(usable_df),
    "n_rows_raw": 9539,
    "generated_by": "01_data_prep.ipynb",
    "columns": {
        "id": "Original ATCOSIM segment identifier",
        "text": "Original lowercase transcript text, unmodified — retained for traceability",
        "split": "Original ATCOSIM train/test split assignment",
        "n_tokens": "Token count of `text`, used for the MIN_TOKENS=4 length filter",
        "callsign": "Parsed callsign portion of the transcript. Derived via keyword-boundary heuristic (split_callsign_instruction_v3). Not manually verified beyond a 15-row spot check.",
        "instruction": "Parsed instruction portion of the transcript (remainder after callsign).",
        "utterance_type": "Parser classification: instruction | callsign_only | non_instruction | ambiguous_no_keyword | empty. Rows outside instruction/callsign_only were excluded from usable_df.",
        "hit_cap": "True if the parser stopped due to the 6-token CALLSIGN_TOKEN_CAP without finding a real boundary (keyword or filler word). Rows with hit_cap=True were excluded from usable_df.",
        "number_tokens": "List of digit-words found in `instruction` (e.g. ['one','three']).",
        "has_numbers": "True if number_tokens is non-empty. Candidate rows for readback-error injection.",
        "instruction_type": "Rule-based classification of the instruction verb: CLIMB, DESCEND, TURN, CONTACT, MAINTAIN, CLEARANCE, IDENTIFY, SQUAWK, REPORT, PROCEED, HOLD, TAXI, SPEED_CHANGE, CROSS, SET_COURSE, or OTHER.",
        "value_type": "Rule-based classification of the numeric value's real-world meaning, inferred from instruction_type: altitude, frequency, heading, squawk_code, or unclassified. Heuristic — not manually validated at scale, spot-checked only.",
        "parser_confidence": "high | medium tier. high = instruction_type resolved AND no cap hit; medium = usable but instruction_type is OTHER. (No 'low' rows present — those were already excluded upstream.)"
    },
    "known_limitations": [
        "~29% of raw ATCOSIM segments excluded: non-instruction utterances, ambiguous no-keyword lines, or unresolved cap-hit boundaries (multi-clause readbacks, route confirmations).",
        "value_type='unclassified' (1,604 rows) reflects instruction types with no single obvious value semantic (IDENTIFY, OTHER, PROCEED, CLEARANCE, REPORT, SPEED_CHANGE, CROSS) — by design, not a parsing failure.",
        "'runway' value_type never populated in this corpus — either genuinely rare in ATCOSIM phraseology or expressed without the literal word 'runway'.",
        "callsign/instruction boundary is a heuristic, not ground truth — validated only via manual spot-checks (~15-30 rows), not exhaustively."
    ]
}

with open(f'{DATA_DIR}/atcosim_enriched_data_dictionary.json', 'w') as f:
    json.dump(data_dictionary, f, indent=2)

print("Data dictionary saved.")
print(json.dumps(data_dictionary, indent=2))

Data dictionary saved.
{
  "dataset": "atcosim_enriched.csv",
  "source": "ATCOSIM corpus (Jzuluaga/atcosim_corpus, HuggingFace)",
  "n_rows": 6519,
  "n_rows_raw": 9539,
  "generated_by": "01_data_prep.ipynb",
  "columns": {
    "id": "Original ATCOSIM segment identifier",
    "text": "Original lowercase transcript text, unmodified \u2014 retained for traceability",
    "split": "Original ATCOSIM train/test split assignment",
    "n_tokens": "Token count of `text`, used for the MIN_TOKENS=4 length filter",
    "callsign": "Parsed callsign portion of the transcript. Derived via keyword-boundary heuristic (split_callsign_instruction_v3). Not manually verified beyond a 15-row spot check.",
    "instruction": "Parsed instruction portion of the transcript (remainder after callsign).",
    "utterance_type": "Parser classification: instruction | callsign_only | non_instruction | ambiguous_no_keyword | empty. Rows outside instruction/callsign_only were excluded from usable_df.",
    "hit_cap"